# RecyMind – Entraînement du premier classifieur (v1)
Personne 2 · MobileNetV3Small, base gelée, dataset Garbage Classification v2 (classes brutes).

**Avant de lancer :** activer le GPU (Exécution → Modifier le type d'exécution → GPU), puis mettre le zip du dataset dans `Drive/recymind/`.

## 1. Connecter Drive et décompresser

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Adapter le nom du zip si besoin
!unzip -q "/content/drive/MyDrive/recymind/garbage.zip" -d /content/data
!ls /content/data

Repérer le dossier qui contient **directement** les sous-dossiers de classes, puis le mettre dans `DATA_DIR` ci-dessous.

## 2. Charger les images

In [ ]:
import tensorflow as tf, json

DATA_DIR = "/content/data/NOM_DU_DOSSIER"   # <-- à remplacer
IMG = (224, 224)
BATCH = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset="training", seed=42,
    image_size=IMG, batch_size=BATCH)
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset="validation", seed=42,
    image_size=IMG, batch_size=BATCH)

class_names = train_ds.class_names
print(class_names, len(class_names))

train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(tf.data.AUTOTUNE)

## 3. Construire le modèle
MobileNetV3 contient déjà son prétraitement : les pixels restent entre 0 et 255 (pas de division par 255).

In [ ]:
base = tf.keras.applications.MobileNetV3Small(
    input_shape=IMG + (3,), include_top=False, weights="imagenet")
base.trainable = False   # on n'entraîne que la tête

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=IMG + (3,)),
    base,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(len(class_names), activation="softmax"),
])

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
model.summary()

## 4. Entraîner

In [ ]:
history = model.fit(train_ds, validation_data=val_ds, epochs=8)

## 5. Accuracy et sauvegarde

In [ ]:
loss, acc = model.evaluate(val_ds)
print(f"Accuracy de validation : {acc:.3f}")

model.save("/content/drive/MyDrive/recymind/modele_v1.keras")
with open("/content/drive/MyDrive/recymind/classes_v1.json", "w") as f:
    json.dump(class_names, f)

## 6. Courbes (pour le rapport)

In [ ]:
import matplotlib.pyplot as plt
plt.plot(history.history["accuracy"], label="entraînement")
plt.plot(history.history["val_accuracy"], label="validation")
plt.xlabel("époque"); plt.ylabel("accuracy"); plt.legend(); plt.show()